<a href="https://colab.research.google.com/github/Malki9/3601763_BD2/blob/main/3601763_BD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Import Libraries

In [1]:
# Import the required libaries
import pandas as pd
import pandas as np
import json
import re

### Load the Dataset

In [2]:
# Create variables and load the .csv datasets
products = pd.read_csv('/content/products.csv')
reviews = pd.read_csv('/content/reviews.csv')
users = pd.read_csv('/content/users.csv')

In [3]:
# Read the file and try to parse it line-by-line manually to avoid 'Trailing data' errors
products_list = []
with open('/content/jcpenney_products.json', 'r') as f:
    for line in f:
        line = line.strip()
        if line:
            try:
                products_list.append(json.loads(line))
            except json.JSONDecodeError:
                # If a single line fails, it might be the start/end of a standard array
                continue

if not products_list:
    # If line-by-line failed, try reading the whole file as one array
    with open('/content/jcpenney_products.json', 'r') as f:
        jcpenney_products = pd.DataFrame(json.load(f))
else:
    jcpenney_products = pd.DataFrame(products_list)

# Load reviewers (usually works fine with lines=True)
jcpenney_reviewers = pd.read_json('/content/jcpenney_reviewers.json', lines=True)

print('Data loaded successfully')

Data loaded successfully


In [4]:
jcpenney_reviewers.head(4)

,Username,DOB,State,Reviewed
0,bkpn1412,31.07.1983,Oregon,[cea76118f6a9110a893de2b7654319c0]
1,gqjs4414,27.07.1998,Massachusetts,[fa04fe6c0dd5189f54fe600838da43d3]
2,eehe1434,08.08.1950,Idaho,[]
3,hkxj1334,03.08.1969,Florida,"[f129b1803f447c2b1ce43508fb822810, 3b0c9bc0be6..."


In [5]:
jcpenney_products.head(4)

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,Reviews,Bought With
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."


In [6]:
jcpenney_products.head(4)

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,Reviews,Bought With
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."


In [7]:
# Checking whether the indexes are correct and no repeating indexes
jcpenney_reviewers.head(4)

,Username,DOB,State,Reviewed
0,bkpn1412,31.07.1983,Oregon,[cea76118f6a9110a893de2b7654319c0]
1,gqjs4414,27.07.1998,Massachusetts,[fa04fe6c0dd5189f54fe600838da43d3]
2,eehe1434,08.08.1950,Idaho,[]
3,hkxj1334,03.08.1969,Florida,"[f129b1803f447c2b1ce43508fb822810, 3b0c9bc0be6..."


In [8]:
reviews.head(4)

,Uniq_id,Username,Score,Review
0,b6c0b6bea69c722939585baeac73c13d,fsdv4141,2,You never have to worry about the fit...Alfred...
1,b6c0b6bea69c722939585baeac73c13d,krpz1113,1,Good quality fabric. Perfect fit. Washed very ...
2,b6c0b6bea69c722939585baeac73c13d,mbmg3241,2,I do not normally wear pants or capris that ha...
3,b6c0b6bea69c722939585baeac73c13d,zeqg1222,0,I love these capris! They fit true to size and...


In [9]:
products.head(4)

,Uniq_id,SKU,Name,Description,Price,Av_Score
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,2.625
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,3.000
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,2.625
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,Youll return to our Alfred Dunner pull-on capr...,41.09,3.500


In [10]:
users.head(4)

,Username,DOB,State
0,bkpn1412,31.07.1983,Oregon
1,gqjs4414,27.07.1998,Massachusetts
2,eehe1434,08.08.1950,Idaho
3,hkxj1334,03.08.1969,Florida


### Data Exploration



In [11]:
# check the dataframes basic information like the datatype
users.info()
jcpenney_products.info()
jcpenney_reviewers.info()
products.info()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Username  5000 non-null   object
 1   DOB       5000 non-null   object
 2   State     5000 non-null   object
dtypes: object(3)
memory usage: 117.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7982 entries, 0 to 7981
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   uniq_id                 7982 non-null   object 
 1   sku                     7982 non-null   object 
 2   name_title              7982 non-null   object 
 3   description             7982 non-null   object 
 4   list_price              7982 non-null   object 
 5   sale_price              7982 non-null   object 
 6   category                7982 non-null   object 
 7   category_tree           7982 non-null   object 
 8   average_product_rati

In [12]:
# Change the datatype of the list_price and the sale_price in the product_review
jcpenney_products['sale_price'] = pd.to_numeric(jcpenney_products['sale_price'], errors='coerce')
jcpenney_products['list_price'] = pd.to_numeric(jcpenney_products['list_price'], errors='coerce')

In [13]:
''' Sometimes the some attributes/ columns has nan which are considered to be string values and hence they are not counted as null. Need to eliminate those as well.
This happens for attributes which are object. Note that an object can be of type list, dictionary, strings. We can see that Reviewed, Reviews, Bought with columns/attributes
in the datasets are type of list, dictionary, list respectively. So we need to identify the list, dictionaries which are empyty '''

# Create a dictionary of dataset
datasets = {
    'users': users,
    'jcpenney_products': jcpenney_products,
    'jcpenney_reviewers': jcpenney_reviewers,
    'products': products,
    'reviews': reviews
}
# Iterate through each column in each dataset and check the datatype
for dataset_name, dataset in datasets.items():
  for column in dataset.columns:
    # Get the columns of the type object
    if dataset[column].dtype == 'object':
      non_null = None
      for value in dataset[column]:
        if value is not None:
          non_null = value
          break

      # Replace the null value with NaN of string column
      if isinstance(non_null, str):
        dataset[column] = (dataset[column].str.strip().replace('', pd.NA))

      # Replace the null value with Nan of list column
      elif isinstance(non_null, list):
        dataset[column] = dataset[column].apply(lambda x: pd.NA if isinstance(x, list) and len(x) == 0 else x)

        # Replace the null values with Nan of Dictionary column
      elif isinstance(non_null, dict):
        dataset[column] = dataset[column].apply(lambda x: pd.NA if isinstance(x, dict) and len(x) == 0 else x)

    # Replace the null values with Nan of the integer and float datatype
    elif dataset[column].dtype in ['int64', 'float64']:
            dataset[column] = dataset[column].replace('', pd.NA)



In [14]:
# Check for null values in each dataframe as a count
# Display in a tabular format
# Function to create the dataframe consist of column and sum
def null_summary(dataset):
    return pd.DataFrame({
        "column": dataset.columns,
        "sum": dataset.isnull().sum().values
    })

# Create summaries of each dataframe
users_null = null_summary(users)
jc_reviews_null = null_summary(jcpenney_reviewers)
jc_products_null = null_summary(jcpenney_products)
products_null = null_summary(products)
reviews_null = null_summary(reviews)

# Put all in dictionary
tables = {
    "users": users_null,
    "jcpenney_reviews": jc_reviews_null,
    "reviews": reviews_null,
    "jc_products": jc_products_null,
    "products": products_null
}

# Find maximum number of rows
max_rows = max(len(dataset) for dataset in tables.values())

# Pad smaller tables
for name, dataset in tables.items():
    if len(dataset) < max_rows:
        padding = pd.DataFrame({
            "column": [""] * (max_rows - len(dataset)),
            "sum": [""] * (max_rows - len(dataset))
        })
        tables[name] = pd.concat([dataset, padding], ignore_index=True)

# Create final aligned table
final_table = pd.concat(tables, axis=1)

final_table

users     jcpenney_reviews        reviews                 jc_products  \
      column sum           column  sum    column sum                  column   
0   Username   0         Username    0   Uniq_id   0                 uniq_id   
1        DOB   0              DOB    0  Username   0                     sku   
2      State   0            State    0     Score   0              name_title   
3                        Reviewed  971    Review   0             description   
4                                                                 list_price   
5                                                                 sale_price   
6                                                                   category   
7                                                              category_tree   
8                                                     average_product_rating   
9                                                                product_url   
10                                                        product_image_urls   
11                                                                     brand   
12                                                      total_number_reviews   
13                                                                   Reviews   
14                                                               Bought With   

             products        
     sum       column   sum  
0      0      Uniq_id     0  
1     67          SKU    67  
2      0         Name     0  
3    543  Description   543  
4   2166        Price  2166  
5    263     Av_Score     0  
6    636                     
7    636                     
8      0                     
9      0                     
10   157                     
11     0                     
12     0                     
13     0                     
14     0

In [15]:
# Standardizing the column names
# Replace the column name 'Bought With' with to bought_with
products.rename(columns={'Bought With': 'bought_with'}, inplace=True)
# Replace the column names with simple
for dataset_name, dataset in datasets.items():
  for column in dataset.columns:
    dataset.rename(columns={column: column.lower()}, inplace=True)

In [16]:
# The users act as the base dataset which is linked with the jcpenney_reviewers and reviews dataset
# Check the Username column in the users, jcpenny_reviews and reviews dataset follows the same standard
# Define the pattern where the first 04 char are simple letters from a-z and the last for as int from 0-9
pattern = r'^[a-z]{4}[0-9]{4}$'
# Get the unique values in each of the dataset of username columns as a Dictionary
users_username = { 'users' : set(users['username']), # convert the username column to set, eliminate the duplicates
                   'reviews': set(reviews['username']),
                   'jcpenney_reviewers' : set(jcpenney_reviewers['username'])
                   }

# Iterate through each value in the users_username
for key, value in users_username.items():
  invalid_found = False
  for username in value:
    if not re.fullmatch(pattern, username):
      print(f" Invalid username in {key}: {username}")
      invalid_found = True
  if not invalid_found: print(f" All usernames in {key} are in the particular standard")

 All usernames in users are in the particular standard
 All usernames in reviews are in the particular standard
 All usernames in jcpenney_reviewers are in the particular standard


In [17]:
# Check all the username in users are there in the jcpenny reviews
users['username'].isin(jcpenney_reviewers['username']).all()

np.True_

In [18]:
''' Accordingly we can see that the users is the base dataset and all the usernames in users are there in jcpenny_reviews. '''
# Check all the username in jcpenny_reviews there in reviews
jcpenney_reviewers['username'].isin(reviews['username']).all()

np.False_

In [19]:
# Get the count of username not there in reviews
jcpenney_reviewers[~jcpenney_reviewers['username'].isin(reviews['username'])]['username'].count()

np.int64(6)

In [20]:
# check the username which are not in the jcpenny_reviews have empty reviews
non_users= jcpenney_reviewers[~jcpenney_reviewers['username'].isin(reviews['username'])][['username','reviewed']]
non_users

,username,reviewed
373,qzsw4431,<NA>
1349,clar2422,<NA>
1562,eevm2113,<NA>
1869,jbym1321,<NA>
2257,eubf2214,<NA>
2410,djzq3441,<NA>


In [21]:
# Drop the rows in jcpenny_reviews where the non existence usernames of review dataset which are not in the jcpenny_reviews
jcpenney_reviewers = jcpenney_reviewers.drop(non_users.index)

In [22]:
# check for duplicate and print the duplicated rows in both users and jcpenny_reviewers dataset
users_duplicate = users[users.duplicated(subset=['username'], keep=False)]
jcpenney_reviewers_duplicate = jcpenney_reviewers[jcpenney_reviewers.duplicated(subset=['username'], keep=False)]
print(users_duplicate)
print(jcpenney_reviewers_duplicate)

      username         dob       state
731   dqft3311  28.07.1995   Tennessee
2619  dqft3311  03.08.1969  New Mexico
      username         dob       state                            reviewed
731   dqft3311  28.07.1995   Tennessee  [5f280fb338485cfc30678998a42f0a55]
2619  dqft3311  03.08.1969  New Mexico  [571b86d307f94e9e8d7919b551c6bb52]


In [23]:
# Drop the second instance in the of the duplicated username from both the dataset
users = users.drop_duplicates(subset='username', keep='first')
jcpenney_reviewers = jcpenney_reviewers.drop_duplicates(subset='username', keep='first')

In [ ]:
# check the usernames in users are the same in jcpenny_reviews usernames with there common column names


In [24]:
# Check the sku in product dataset is present in the jcpenny_products
products['sku'].isin(jcpenney_products['sku']).all()

np.False_

In [25]:
# check sku in products is unique
products['sku'].is_unique

False

In [26]:
# check uniq_id in review is unique
reviews['uniq_id'].is_unique

False

In [25]:
# Check the unique_id in review dataset is there in the product dataset
reviews['uniq_id'].isin(products['uniq_id']).all()


np.True_

In [26]:
# Check the unique_id in review dataset is there in the jc_penny dataset
reviews['uniq_id'].isin(jcpenney_products['uniq_id']).all()

np.True_

In [27]:
# Check the unique_id in jcpenny is there in the review
jcpenney_products['uniq_id'].isin(reviews['uniq_id']).all()

np.True_

In [28]:
# Check unique_id is unique in products dataset and the jcpenny_product
products['uniq_id'].is_unique

True

In [29]:
jcpenney_products['uniq_id'].is_unique

True

In [30]:
# Map the column pairs in jcpenny_product and products dataframe
columns = [('list_price','price'),
('sku','sku')]

for jcpenny_column, product_column in columns:
  # Get the uniq_id
  uniq_id = jcpenney_products.loc[jcpenney_products[jcpenny_column].isnull(), 'uniq_id']
  #print(len(uniq_id))
  # Check for ids match
  match_id = products.loc[(products['uniq_id'].isin(uniq_id) & (products[product_column].isnull()))]
  # Drop the rows in both dataset
  jcpenney_products = jcpenney_products.drop(match_id.index)
  products = products.drop(match_id.index)


In [32]:
final_table

users     jcpenney_reviews        reviews                 jc_products  \
      column sum           column  sum    column sum                  column   
0   username   0         username    0   uniq_id   0                 uniq_id   
1        dob   0              dob    0  username   0                     sku   
2      state   0            state    0     score   0              name_title   
3                        reviewed  965    review   0             description   
4                                                                 list_price   
5                                                                 sale_price   
6                                                                   category   
7                                                              category_tree   
8                                                     average_product_rating   
9                                                                product_url   
10                                                        product_image_urls   
11                                                                     brand   
12                                                      total_number_reviews   
13                                                                   reviews   
14                                                               bought with   

            products      
    sum       column sum  
0     0      uniq_id   0  
1     0          sku   0  
2     0         name   0  
3     0  description   0  
4     0        price   0  
5     2     av_score   0  
6   520                   
7   520                   
8     0                   
9     0                   
10    1                   
11    0                   
12    0                   
13    0                   
14    0

In [33]:
# Get the rows with null category
jcpenney_products[jcpenney_products['category'].isnull()]

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,reviews,bought with
587,af49185dcfd709b8a21f1b2fdf408cb9,pp5006510551,Xersion™ Go Swiftly Vest - Plus,Wear our vest for exercise or errands. It feat...,-65.27,32.62,<NA>,<NA>,1.600000,http://www.jcpenney.com/xersion-go-swiftly-ves...,http://s7d9.scene7.com/is/image/JCPenney/DP012...,Xersion,5,"[{'User': 'qdpf1442', 'Review': 'Bought for my...","[9457adbc3eb6d97f4578960a2846db1a, a9ec1584a29..."
588,d98546105db42d9fb4b3d5f1ff0c1b3b,pp5006380890,Xersion™ Graphic Leggings - Tall,A splash of graphic excitement takes our Xersi...,-58.01,33.83,<NA>,<NA>,2.000000,http://www.jcpenney.com/xersion-graphic-leggin...,http://s7d9.scene7.com/is/image/JCPenney/DP120...,Xersion,1,"[{'User': 'psmw1224', 'Review': 'Very comforta...","[26ea12aec56b9bc8c16046bdb976b6f3, 5cae6711501..."
589,d2cef6ebd8a99bf7aa7d1d7bc0583c06,pp5006380890,Xersion™ Graphic Leggings - Tall,A splash of graphic excitement takes our Xersi...,-58.01,33.83,<NA>,<NA>,5.000000,http://www.jcpenney.com/xersion-graphic-leggin...,http://s7d9.scene7.com/is/image/JCPenney/DP120...,Xersion,1,"[{'User': 'ybuy2431', 'Review': 'Very comforta...","[70533fb454af2611b97dea0854ea1289, 66c83cdc773..."
590,7ab9a41c73121f0c90b4641fe906f62e,pp5006380890,Xersion™ Graphic Leggings - Tall,A splash of graphic excitement takes our Xersi...,-58.01,33.83,<NA>,<NA>,4.000000,http://www.jcpenney.com/xersion-graphic-leggin...,http://s7d9.scene7.com/is/image/JCPenney/DP120...,Xersion,1,"[{'User': 'gbjg1333', 'Review': 'Very comforta...","[2212e6c0ec2a4b34c1d99c2b537a8f69, c59b259353a..."
592,aae74d7df3f8247302ff4a8bed07fa96,pp5006350688,Xersion™ Graphic Tank Top - Girls 7-16 and Plus,"With bright color and easy comfort, this tank ...",-21.76,10.87,<NA>,<NA>,2.777778,http://www.jcpenney.com/xersion-graphic-tank-t...,http://s7d9.scene7.com/is/image/JCPenney/DP030...,Xersion,9,"[{'User': 'kvnf1331', 'Review': 'My daughter i...","[b536148d689181aa2e7bfdf844bc13ef, 4d51bed9cc7..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7429,2a7d0a6f29ac538add710758c157e421,pp5006311155,"GE Profile™ 36"" High Performance Range Hood",true This high-performance range hood features...,458.07,412.14,<NA>,<NA>,2.125000,http://www.jcpenney.com/ge-profile-series-36-h...,http://s7d9.scene7.com/is/image/JCPenney/DP012...,GE,8,"[{'User': 'imzz2343', 'Review': 'It came with ...","[afb41dde103bfc06d482c402e287b23b, f9730f00b95..."
7430,e93875c8457df78a8d3b23ff2c4871ea,pp5006311162,"GE Profile™ 36"" High Performance Range Hood",true This high-performance range hood’s 390-CF...,627.28,564.43,<NA>,<NA>,4.000000,http://www.jcpenney.com/ge-profile-series-36-h...,http://s7d9.scene7.com/is/image/JCPenney/DP112...,GE,8,"[{'User': 'hkvs2324', 'Review': 'Love the feat...","[c0881d5a8eb53e2c55e85fad94f7a393, 2dfc28dd7fd..."
7431,60f75c38dfce8ae5fb356eac8e2a982a,pp5006310426,GE® Space-Saving 2.6 DOE Capacity Top-Loading ...,true This portable washer from GE® goes just a...,1026.12,923.39,<NA>,<NA>,2.500000,http://www.jcpenney.com/ge-space-saving-26-doe...,http://s7d9.scene7.com/is/image/JCPenney/DP111...,GE,8,"[{'User': 'tdda1422', 'Review': 'I bought this...","[eaa321e1062c5b09da4188d2e0675ead, cfbf109a71c..."
7432,98b587f7a9f7d101c7638cbf925c28ad,pp5006310427,GE® Space-Saving 2.6 DOE Capacity Top-Loading ...,true This compact washer from GE® fits just ab...,1026.12,923.39,<NA>,<NA>,3.375000,http://www.jcpenney.com/ge-space-saving-26-doe...,http://s7d9.scene7.com/is/image/JCPenney/DP111...,GE,8,"[{'User': 'mdck4414', 'Review': 'I bought this...","[8b357637ac6d050e10ea904b4a31ac03, 455a371d9db..."


In [36]:
# Get the statistics of object column
jcpenney_products.describe(include='object')

,uniq_id,sku,name_title,description,category,category_tree,product_url,product_image_urls,brand,reviews,bought with
count,5767,5767,5767,5767,5247,5247,5767,5766,5767,5767,5767
unique,5767,4406,4267,4290,934,1597,5767,4713,544,5767,5767
top,7a8a7cba7b69b4c46ceac1d9666d84f0,pp5006210554,Champion® Vapor Shorts - Big & Tall,Stay comfortable and looking great all day wit...,view all,jcpenney|view all,http://www.jcpenney.com/hoover-windtunnel-upri...,http://s7d9.scene7.com/is/image/JCPenney/DP101...,ARIZONA,"[{'User': 'qven4314', 'Review': '1) the hose i...","[fc831da1b1d09b0bd7d6893e9615232f, 9f5f2c625d4..."
freq,1,18,23,30,294,151,1,18,333,1,1


In [39]:
# Count for each uniq_id how many sku are there and print the uniq_id which has more than one sku
uniq_id_counts = jcpenney_products['uniq_id'].value_counts()
uniq_id_with_multiple_skus = uniq_id_counts[uniq_id_counts > 1]

In [40]:
uniq_id_with_multiple_skus

,count
uniq_id,


In [41]:
# get the rows with sku = 'pp5006380337'
jcpenney_products[jcpenney_products['sku'] == 'pp5006380337']

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,reviews,bought with
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."
4,d969a8542122e1331e304b09f81a83f6,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.125,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'nbmi2334', 'Review': 'You never hav...","[0ca5ad2a218f59eb83eec1e248a0782d, 9869fc8da14..."
5,cf73bb2bd93bbd6e1bdf48d399992270,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,deals,jcpenney|shops|deals,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'mewj3321', 'Review': 'You never hav...","[fa181e36fe1286e311b2fc92e4db1fa8, f219d07e6d7..."


In [42]:
# check sku in products dataframe is unique
products['sku'].is_unique

False

In [35]:
jcpenney_products.head(4)

,uniq_id,sku,name_title,description,list_price,sale_price,category,category_tree,average_product_rating,product_url,product_image_urls,brand,total_number_reviews,reviews,bought with
0,b6c0b6bea69c722939585baeac73c13d,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'fsdv4141', 'Review': 'You never hav...","[898e42fe937a33e8ce5e900ca7a4d924, 8c02c262567..."
1,93e5272c51d8cce02597e3ce67b7ad0a,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,alfred dunner,jcpenney|women|alfred dunner,3.000,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'tpcu2211', 'Review': 'You never hav...","[bc9ab3406dcaa84a123b9da862e6367d, 18eb69e8fc2..."
2,013e320f2f2ec0cf5b3ff5418d688528,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,2.625,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'pcfg3234', 'Review': 'You never hav...","[3ce70f519a9cfdd85cdbdecd358e5347, b0295c96d2b..."
3,505e6633d81f2cb7400c0cfa0394c427,pp5006380337,Alfred Dunner® Essential Pull On Capri Pant,You'll return to our Alfred Dunner pull-on cap...,41.09,24.16,view all,jcpenney|women|view all,3.500,http://www.jcpenney.com/alfred-dunner-essentia...,http://s7d9.scene7.com/is/image/JCPenney/DP122...,Alfred Dunner,8,"[{'User': 'ngrq4411', 'Review': 'You never hav...","[efcd811edccbeb5e67eaa8ef0d991f7c, 7b2cc00171e..."


In [42]:
# Remove null values in list_price in jcpenny_products and products dataset
# Get the rows in the jcpenny_products sku is null
jcpenny_sku_null = jcpenney_products[jcpenney_products['list_price'].isnull()]

# Get uniq_id
uniq_id = jcpenny_sku_null['uniq_id']
uniq_id

# Check the uniq_id in the product dataset
#match_id = products[products['uniq_id'].isin(uniq_id) & (products['sku'].isnull())]
#match_id

,uniq_id
7,4d9337e3c8f974d3c420cdc5c58b3fc3
8,44f8f8f108c6856acf9630dd1d78516d
9,8d1d057f5f808c10ce243c222ab0ef6e
10,f3e02c48f16b56e8c1f126c8fe762812
15,8149f96381ad38bc848c1c12b7ad3228
...,...
7964,e7dfc34b04500be1ab450f7a94a64aab
7965,1b5d3b8248a83c47ac21cffd60efffcb
7966,cb3304106ae99247bb3223270f3c56b1
7980,9887bb20c12be3c09d094e97ac8d22be


This confirms that the uniq_id columns in the products and jcpenny_products does not have multiple sku.

This confirms that jcpenny_products is the main dataset which contains all the details of the particular product the product_id, sku, name, description and the ratings

In [ ]:
# Iterate through each column in the jcpenney product and identify whether the product details (such as SKU) are same as the products using Unique_id
for column in products.columns:
  # Check the iterated column is uniq_id
  if column == 'Uniq_id':
    unmatch = False
    # Iterate through each value in the products
    for uniq_id in jcpenney_products['uniq_id']:
      if uniq_id not in products['Uniq_id'].values:
        print(f"Unique_id {uniq_id} not found in products")
        unmatch = True

    if not unmatch:
      print(f"All the Unique_id found in products are there in the Jcpenny_product.json file")

  elif column == 'sku':
    unmatch = False
    # Iterate through each value in the
    for product_name in products['SKU']:
      if product_name not in jcpenney_products['sku'].values:
        print(f"SKU {product_name} not found in jcpenney_products")
        unmatch = True

    if not unmatch:
      print(f"All the SKU found in jcpenney_products are there in the producsts csv")


In [ ]:
for uniq_id in jcpenney_products['uniq_id']:

    # Check if uniq_id exists in products
    if uniq_id in products['Uniq_id'].values:

        # Get SKU from both datasets
        sku_products = products.loc[products['Uniq_id'] == uniq_id, 'SKU'].values[0]
        sku_jc = jcpenney_products.loc[jcpenney_products['uniq_id'] == uniq_id, 'sku'].values[0]

        # Compare SKU
        if sku_products != sku_jc:
            print(f"Mismatch for Unique_id {uniq_id}:")
            print(f"   products SKU = {sku_products}")
            print(f"   jcpenney SKU = {sku_jc}")

    else:
        print(f"Unique_id {uniq_id} not found in products")

In [ ]:
# Check Username is consitent over users dataframe and jcpenny_reviewers
# Define a dictionary variable and assign the columns as set
users_username = {
    'users' : set(users['Username']), # convert the username column to set, eliminates the duplicates
    'jcpenney_reviewers' : set(jcpenney_reviewers['Username'])
}

# Intersect to get the common values
username_common = set.intersection(*users_username.values())
print(f'The number of users common among the two dataset : {len(username_common)}')


All the users in user.csv exists in jcpenny_reviews
